# Project 3 -- Niharika Rai

**TA Help:** (for instance) John Smith, Alice Jones, etc., list names of any TAs who helped you

- For example: Help with figuring out how to write a function (describe the tasks that they helped you with)

**Collaboration:** My Friend in CS, My Uncle, Another Student, etc., list names of any other people who helped you

(describe the tasks that they helped you with)
- For example: helped figuring out how to load the dataset.
- Another example: helped debug error with my plot.

**Internet Resources:** Stack Exchange, Stack Overflow, etc.

(describe any information that you learned from internet resources, including the URLs)
- data frames in Pandas versus R from StackOverflow  https://stackoverflow.com/questions/8991709/why-were-pandas-merges-in-python-faster-than-data-table-merges-in-r-in-2012

**ChatGPT, Gemini, Claude, etc:** Any language models or generative AI chatbots that helped you.

(if you used any such tools, please tell us here)
- For example:  I asked ChatGPT how to define a new data frames
- Another example:  Gemini told me how to make a function for sorting my data

- ***Link to AI Chat History***: Here is the link to AI Chat History: https://chatgpt.com/share/698c109c-1e18-800d-858a-94b0a209f25b Please share a link to your chat if you used AI (ex. ChatGPT Shared Links)
**OVERALL MESSAGE:** Any time that you used anything except your brain to solve the questions in these projects, you need to disclose such resources at the start of the project, with details about your usage of the tools.

**YOUR OWN WORK:** Even when you utilize other resources, do NOT just copy and paste.  Write all explanations in your own words, using several sentences in English, which are understandable and which you wrote (and did not just copy and paste).

## Question 1

In [1]:
import requests
from lxml import html
import time

def scrape_with_delay(url, delay=1):
    """Scrape a URL with a delay between requests."""
    response = requests.get(url)
    time.sleep(delay)  # Wait before next request
    return response

# Example: Scraping multiple pages with delays
base_url = "https://quotes.toscrape.com"
pages_to_scrape = [
    f"{base_url}/page/1/",
    f"{base_url}/page/2/",
    f"{base_url}/page/3/",
    f"{base_url}/page/4/",
    f"{base_url}/page/5/"
]

# List to store all quotes from all pages
all_quotes = []

start_time = time.perf_counter()   # start total timer

# Loop over each page in the list
for page_url in pages_to_scrape:
    print(f"Scraping {page_url}...")

    # HTTP request to page_url
    response = scrape_with_delay(page_url, delay=2)
    tree = html.fromstring(response.text)

    # Extract quotes
    quotes = tree.xpath('//span[@class="text"]/text()')
    all_quotes.extend(quotes)

    print(f"  Found {len(quotes)} quotes")

end_time = time.perf_counter()     # end total timer

total_time = end_time - start_time
avg_time_per_request = total_time / len(pages_to_scrape)

print(f"\nTotal quotes collected: {len(all_quotes)}")
print(f"Total scraping time: {total_time:.2f} seconds")
print(f"Average time per request: {avg_time_per_request:.2f} seconds")

Scraping https://quotes.toscrape.com/page/1/...
  Found 10 quotes
Scraping https://quotes.toscrape.com/page/2/...
  Found 10 quotes
Scraping https://quotes.toscrape.com/page/3/...
  Found 10 quotes
Scraping https://quotes.toscrape.com/page/4/...
  Found 10 quotes
Scraping https://quotes.toscrape.com/page/5/...
  Found 10 quotes

Total quotes collected: 50
Total scraping time: 10.68 seconds
Average time per request: 2.14 seconds


1.1 I used the sample code from the question to write a scraper with rate limiting (2-second delay). 
1.2 I modified the sample code from the question to add pages 4 and 5, so that I am scrapting atleast 5 pages. 
1.3 I used ChatGPT to debug my code and get the correct time functions for calculating the total scraping time. My initial code was incorrect and I was getting errors. To get the total starting time, I had to first find the start and end time and then find their difference. Both the start and end time were found by applying the 'perf_counter()' function to time. After debugging the code, I found that the total scraping time was 10.68 seconds rounded to two decimal places.  
1.4 I used ChatGPT to debug and calculate the average time per request. My initial code, that I contructed using the sample code's logic, was giving me errors. The average time was calculated by diving the total scraping time by the number of pages scraped. After debugging my code, I found that the average time per request was 2.14 seconds rounded to two decimal places.  
1.5 The explanation was not mentioned in the question, so I used ChatGPT to find the answer. Rate-limiting can be observed through the delays that we wrote in our function. These delays help to pause the execution of requests for a specified number of seconds (eg: 1-2 seconds). The main benefits that make rate limiting important is that it protects the server (by preventing its overload), prevents being blocked by websites, ensures fair usage, avoids extra costs, and improves the reliability of requests. 

## Question 2

In [2]:
import random

def scrape_with_exponential_backoff(url, max_retries=5, initial_delay=1):
    """
    Scrape a URL with exponential backoff retry logic.

    If we get an error, wait longer before retrying:
    - First retry: wait 1 second
    - Second retry: wait 2 seconds
    - Third retry: wait 4 seconds
    - etc. (exponential: 2^retry_number)
    """
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=10)

            # Check for rate limiting errors
            if response.status_code == 429:
                # Use the exponential backoff delay formula (see docstring)
                wait_time = initial_delay * (2 ** attempt)

                # Just for us to see what is going on:
                print(f"Rate limited! Waiting {wait_time} seconds before retry {attempt + 1}...")

                # Give some wait time, based on our calculated delay time
                time.sleep(wait_time)
                continue

            # Check for server errors
            if response.status_code >= 503:
                wait_time = initial_delay * (2 ** attempt)
                print(f"Server error {response.status_code}! Waiting {wait_time} seconds before retry {attempt + 1}...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()  # Raises exception for 4xx/5xx errors
            return response

        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                # Last attempt failed
                raise Exception(f"Failed after {max_retries} attempts: {e}")

            wait_time = initial_delay * (2 ** attempt)

            # Add some randomness - this generates random number between a and b, allowing us to add extra delay time
            extra_delay = random.uniform(0, 0.1 * wait_time)
            # The resulting number is added to sleept time, to prevent numerous retries simultaneously
            total_wait = wait_time + extra_delay

            print(f"Error: {e}")
            print(f"Waiting {total_wait:.2f} seconds before retry {attempt + 1}...")
            time.sleep(total_wait)
    raise Exception("Max retries exceeded")

In [3]:
# Example usage
url = "https://quotes.toscrape.com/page/1/"
response = scrape_with_exponential_backoff(url)
tree = html.fromstring(response.text)
quotes = tree.xpath('//span[@class="text"]/text()')
print(f"Successfully scraped {len(quotes)} quotes")

Successfully scraped 10 quotes


In [4]:
# Example: Scraping multiple pages using robust scraper created above.
base_url = "https://quotes.toscrape.com"
pages_to_scrape = [
    f"{base_url}/page/1/",
    f"{base_url}/page/2/",
    f"{base_url}/page/3/",
    f"{base_url}/page/4/",
    f"{base_url}/page/5/"
]

# List to store all quotes from all pages
all_quotes = []

# Loop over each page in the list
for page_url in pages_to_scrape:
    print(f"Scraping {page_url}...")

    # HTTP request to page_url
    response = scrape_with_exponential_backoff(url)
    tree = html.fromstring(response.text)

    # Extract quotes
    quotes = tree.xpath('//span[@class="text"]/text()')
    all_quotes.extend(quotes)

    print(f"  Found {len(quotes)} quotes")

print(f"\nSuccessfully scraped {len(all_quotes)} quotes")

Scraping https://quotes.toscrape.com/page/1/...
  Found 10 quotes
Scraping https://quotes.toscrape.com/page/2/...
  Found 10 quotes
Scraping https://quotes.toscrape.com/page/3/...
  Found 10 quotes
Scraping https://quotes.toscrape.com/page/4/...
  Found 10 quotes
Scraping https://quotes.toscrape.com/page/5/...
  Found 10 quotes

Successfully scraped 50 quotes


2.1 I implemented the exponential backoff function using the sample code in the question.
2.2 I handled the HTTP error codes 429 and 503 using the sample code in the question. However, the sample code had the service error written as 500, but the question asked for 503, so I replaced 500 with 503 to meet the question's requirements. 
2.3 I added jitter to avoid synchronized retries. This was implemented by generating random numbers between 0 and 0.1 * wait_time, which allows us to add extra delay time to the sleep time and prevent numerous entries simultaneously. 
2.4 I tested the function using the example sample code given in the question. This sample code had the try and except statements to efficiently handle errors.  
2.5 I used parts of the sample code, given for question 1, and modified it to implement the exponential backoff function for the response variable. This helped me to scrape multiple pages using the robust scraper because it now used the exponential backoff function.  

## Question 3

In [5]:
import random
import requests
from lxml import html

# Common User-Agent strings
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
]

base_url = "https://quotes.toscrape.com"
pages_to_scrape = [
    f"{base_url}/page/1/",
    f"{base_url}/page/2/",
    f"{base_url}/page/3/",
    f"{base_url}/page/4/",
    f"{base_url}/page/5/"
]

    
def scrape_with_headers(session, url):
    """Scrape with custom headers and rotating User-Agent."""
    user_agent = random.choice(user_agents)

    headers = {
        'User-Agent': user_agent,
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Accept-Encoding': 'gzip, deflate',
        'Connection': 'keep-alive',
    }

    print(f"Using User-Agent: {user_agent}")
    response = session.get(url, headers=headers)
    return response


session = requests.Session()

all_quotes = []

for page_url in pages_to_scrape:
    print(f"Scraping {page_url}...")
    
    response = scrape_with_headers(session, page_url)

    tree = html.fromstring(response.text)
    quotes = tree.xpath('//span[@class="text"]/text()')
    all_quotes.extend(quotes)

    print(f"Found {len(quotes)} quotes")
    print(f"Cookies so far: {session.cookies}\n")

print(f"Scraped: {len(all_quotes)} quotes with custom headers")

Scraping https://quotes.toscrape.com/page/1/...
Using User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36
Found 10 quotes
Cookies so far: <RequestsCookieJar[]>

Scraping https://quotes.toscrape.com/page/2/...
Using User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36
Found 10 quotes
Cookies so far: <RequestsCookieJar[]>

Scraping https://quotes.toscrape.com/page/3/...
Using User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36
Found 10 quotes
Cookies so far: <RequestsCookieJar[]>

Scraping https://quotes.toscrape.com/page/4/...
Using User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36
Found 10 quotes
Cookies so far: <RequestsCookieJar[]>

Scraping https://quotes.tosc

3.1 I implemented a scraper with custom User-Agent headers using sample code given in the question. 
3.2 I used a session object to maintain cookies using the sample code given in the question.
3.3 I ensured that I was rotating User-Agent strings across requests by using the random.choice() function on the user agents. This ensured that the user agents would be picked randomly. I initially had a doubt about this and used ChatGPT to find that random.choice() would rotate the user-agent strinfs across requests.  
3.4 I used parts of codes from the previous questions to scrape multiple pages using the session.
3.5 Custom headers such as "User-Agents" might be necessary to help the scraper mimic normal user behavior and reduce the likelihood of being blocked. This is because some websites check the "User-Agent" header to identify what type of browser or program is making the request, often leading those sites to block requests that don’t have a proper User-Agent. Hence, customizing our request headers makes them appear more like a regular browser and prevents this blocking. 

## Question 4

In [8]:
from urllib.robotparser import RobotFileParser
from urllib.parse import urljoin, urlparse

def check_robots_txt(base_url, user_agent='*'):
    """Check robots.txt for a given URL."""
    robots_url = urljoin(base_url, '/robots.txt')

    try:
        # Send HTTP GET request to '/robots.txt'
        # Because we have 'timeout=5', exception occurs if we do not get a response back in 5 seconds
        response = requests.get(robots_url, timeout=5)

        # 200 denotes successful request to server
        if response.status_code == 200:
            rp = RobotFileParser()
            rp.set_url(robots_url)
            rp.read()

            # Check if we can access a specific path

            # Break URL down
            parsed_url = urlparse(base_url)

            # Extract the path part. If there is no path, the default is '/'
            path = parsed_url.path or '/'

            # Check permission
            can_fetch = rp.can_fetch(user_agent, base_url)
            print(f"robots.txt found at: {robots_url}")
            print(f"Can fetch {base_url}? {can_fetch}")

            # Check if the site specified a delay, and if so we show it
            crawl_delay = rp.crawl_delay(user_agent)
            if crawl_delay:
                print(f"Crawl delay specified: {crawl_delay} seconds")

            # Return parsed rules and permission result
            return rp, can_fetch

        # Missing robots.txt case:
        else:
            print(f"robots.txt not found or inaccessible (status: {response.status_code})")
            return None, True  # If no robots.txt, generally we assume scraping is allowed
    except Exception as e:
        print(f"Error checking robots.txt: {e}")

        # Returns True as robots.txt could not be checked, and proceed with scraping. But, we take into consideration that it was only assumed.
        return None, True

In [9]:
# Example 1: Check robots.txt
base_url = "https://quotes.toscrape.com"

# Call the function
rp, allowed = check_robots_txt(base_url)

# allowed == True
if allowed:
    print("\nProceeding with scraping...")
    # Your scraping code here
else:
    print("\nScraping not allowed by robots.txt!")

robots.txt not found or inaccessible (status: 404)

Proceeding with scraping...


In [10]:
# Example 2: 

base_url = "https://books.toscrape.com/"

# Call the function
rp, allowed = check_robots_txt(base_url)

# allowed == True
if allowed:
    print("\nProceeding with scraping...")
    # Your scraping code here
else:
    print("\nScraping not allowed by robots.txt!")

robots.txt not found or inaccessible (status: 404)

Proceeding with scraping...


In [12]:
# Example 3: 

base_url = "https://www.google.com/"

# Call the function
rp, allowed = check_robots_txt(base_url)

# allowed == True
if allowed:
    print("\nProceeding with scraping...")
    # Your scraping code here
else:
    print("\nScraping not allowed by robots.txt!")

robots.txt found at: https://www.google.com/robots.txt
Can fetch https://www.google.com/? False

Scraping not allowed by robots.txt!


4.1 I wrote a function to check and parse robots.txt using the sample code given in the question.
4.2 The crawl delays from robots.txt are checked and displayed if the delays exist. This was already present in the sample code. 
4.3 I also check if specific paths are allowed before scraping. This can be seen in breaking the url and extracting its path part. This was also already present in the sample code.   
4.4 I tested my function on multiple websites listed in the question. I replicated the format of example 1, and replaced the urls each time and ran the function to test it. 
4.5 Respecting robots.txt is important because it is an implementation of the Robots Exclusion Protocol, that defines a set of rules for web crawlers and web robots on what sections of a website they may or may not visit. Hence, it basically is a standard that websites use to tell scrapers which parts of the site they can and cannot access. 

## Question 5

In [34]:
import time
import logging
import requests
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser
from lxml import html

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class RobustScraper:
    """Web scraper with rate limiting, error handling, and robots.txt checking."""

    def __init__(self, base_url, user_agent=None, default_delay=1):
        self.base_url = base_url
        self.session = requests.Session()
        self.default_delay = default_delay
        self.robots_parser = None

        # Set headers
        headers = {
            'User-Agent': user_agent or 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        }
        self.session.headers.update(headers)

        # Load robots.txt
        self._check_robots_txt()

    def _check_robots_txt(self):
        """Check and parse robots.txt."""
        robots_url = urljoin(self.base_url, '/robots.txt')
        try:
            response = self.session.get(robots_url, timeout=5)
            if response.status_code == 200:
                self.robots_parser = RobotFileParser()
                self.robots_parser.set_url(robots_url)
                self.robots_parser.read()
                logger.info(f"Loaded robots.txt from {robots_url}")
        except Exception as e:
            logger.warning(f"Could not load robots.txt: {e}")

    def _can_fetch(self, url):
        """Check if URL can be fetched according to robots.txt."""
        if self.robots_parser:
            return self.robots_parser.can_fetch(self.session.headers['User-Agent'], url)
        return True

    def _get_crawl_delay(self):
        """Get crawl delay from robots.txt."""
        if self.robots_parser:
            delay = self.robots_parser.crawl_delay(self.session.headers['User-Agent'])
            if delay:
                return delay
        return self.default_delay

    def fetch(self, url, max_retries=3, initial_delay=1):
        """Fetch a URL with exponential backoff and error handling."""
        if not self._can_fetch(url):
            logger.warning(f"robots.txt disallows: {url}")
            return None

        for attempt in range(max_retries):
            try:
                # Respect crawl delay
                time.sleep(self._get_crawl_delay())

                response = self.session.get(url, timeout=10)

                # Handle rate limiting
                if response.status_code == 429:
                    logger.warning("Rate limited (429). Backing off...")
                    time.sleep(initial_delay * (2 ** attempt))
                    continue

                # Handle server errors
                if response.status_code >= 500:
                    logger.warning(f"Server error {response.status_code}. Retrying...")
                    time.sleep(initial_delay * (2 ** attempt))
                    continue

                response.raise_for_status()
                logger.info(f"Successfully fetched {url}")
                return response

            except requests.exceptions.RequestException as e:
                if attempt == max_retries - 1:
                    logger.error(f"Failed to fetch {url} after {max_retries} attempts: {e}")
                    return None
                else:
                    backoff = initial_delay * (2 ** attempt)
                    logger.warning(f"Request failed ({e}). Retrying in {backoff}s...")
                    time.sleep(backoff)

        return None

    def scrape_quotes(self, url):
        """Scrape quotes from a page."""
        response = self.fetch(url)
        if not response:
            return []
        tree = html.fromstring(response.text)
        quotes = []

        quote_containers = tree.xpath('//div[@class="quote"]')
        for container in quote_containers:
            text = container.xpath('.//span[@class="text"]/text()')
            author = container.xpath('.//small[@class="author"]/text()')

            if text and author:
                quotes.append({
                    'text': text[0],
                    'author': author[0]
                })

        logger.info(f"Scraped {len(quotes)} quotes from {url}")
        return quotes

# Testing the scraper on multiple pages.
base_url = "https://quotes.toscrape.com"
scraper = RobustScraper(base_url)

pages_to_scrape = [
    f"{base_url}/page/1/",
    f"{base_url}/page/2/",
    f"{base_url}/page/3/",
    f"{base_url}/page/4/",
    f"{base_url}/page/5/"
]

all_quotes = []

for page_url in pages_to_scrape:
    logger.info(f"Scraping page: {page_url}")
    page_quotes = scraper.scrape_quotes(page_url)
    all_quotes.extend(page_quotes)

print(f"\nScraped {len(all_quotes)} quotes total:")
for quote in all_quotes[:]:
    print(f"  - {quote['text'][:50]}... - {quote['author']}")

2026-02-10 23:52:34,679 - INFO - Scraping page: https://quotes.toscrape.com/page/1/
2026-02-10 23:52:35,712 - INFO - Successfully fetched https://quotes.toscrape.com/page/1/
2026-02-10 23:52:35,713 - INFO - Scraped 10 quotes from https://quotes.toscrape.com/page/1/
2026-02-10 23:52:35,714 - INFO - Scraping page: https://quotes.toscrape.com/page/2/
2026-02-10 23:52:36,748 - INFO - Successfully fetched https://quotes.toscrape.com/page/2/
2026-02-10 23:52:36,749 - INFO - Scraped 10 quotes from https://quotes.toscrape.com/page/2/
2026-02-10 23:52:36,750 - INFO - Scraping page: https://quotes.toscrape.com/page/3/
2026-02-10 23:52:37,782 - INFO - Successfully fetched https://quotes.toscrape.com/page/3/
2026-02-10 23:52:37,783 - INFO - Scraped 10 quotes from https://quotes.toscrape.com/page/3/
2026-02-10 23:52:37,784 - INFO - Scraping page: https://quotes.toscrape.com/page/4/
2026-02-10 23:52:38,816 - INFO - Successfully fetched https://quotes.toscrape.com/page/4/
2026-02-10 23:52:38,817 - IN


Scraped 50 quotes total:
  - “The world as we have created it is a process of o... - Albert Einstein
  - “It is our choices, Harry, that show what we truly... - J.K. Rowling
  - “There are only two ways to live your life. One is... - Albert Einstein
  - “The person, be it gentleman or lady, who has not ... - Jane Austen
  - “Imperfection is beauty, madness is genius and it'... - Marilyn Monroe
  - “Try not to become a man of success. Rather become... - Albert Einstein
  - “It is better to be hated for what you are than to... - André Gide
  - “I have not failed. I've just found 10,000 ways th... - Thomas A. Edison
  - “A woman is like a tea bag; you never know how str... - Eleanor Roosevelt
  - “A day without sunshine is like, you know, night.”... - Steve Martin
  - “This life is what you make it. No matter what, yo... - Marilyn Monroe
  - “It takes a great deal of bravery to stand up to o... - J.K. Rowling
  - “If you can't explain it to a six year old, you do... - Albert Einstein
  -

5.1 I created a RobustScraper class with all advanced features. This was the most confusing part because I had to compile all the defined functions under a common class called RobustScraper. Although the sample codes from the questions helped, they were somewhat misleading and leading to errors. I realized that the '...' meant that the code was incomplete and felt lost with what to put here and how to structure them, so I used ChatGPT to understand the code lines. These code lines are all parts of what I did for the previous questions. But with the init function, I found the code a little hard to structure and used ChatGPT to fill in the correct syntax to avoid code breakage and syntax errors.   
5.2 The robots.txt checking was implemented in the check_robots_txt function. It includes the code pieces from the previous question, but has 'self' written in front of the minor function used the check and parse the file. 
5.3 I first set up the logging using the sample code in the question. Then, I used this logging in the fetch function to track scraping activity. Part of the code lines were given in the sample code, but I used ChatGPT to add the logger lines to handle service errors: 429 and 500. These specific errors were already included in the sample code so I did not change the 500 to 503 like I did for question 2.   
5.4 I tested the scraper on multiple pages and used ChatGPT with my for loop debugging since my code had errors. I displayed the total number of quotes scraped, from all of the pages, and displayed all the quotes along with their authors to ensure that I scraped all 5 pages. 
5.5 I demonstrated error handling and retry logic using try and except in my code. This helped me to check and parse robots.txt, respect crawl delay, rate limiting, service errors, and request retries. 
5.6 The ethical considerations of web scraping are to be able to follow all the guidelines and policies of any given website. For example, we must follow the robots.txt rules, avoid overloading servers through frequent request rates, and avoid collecting private user data. As a general guideline, we as users must verify our identities through user agents, follow rate limits, and follow the website's terms of service. Following these ethical considerations enable us to be ethical users since we are not disrupting services, bypassing access controls and violating privacy rights. I got this explanation from ChatGPT but summarized it in my own words to showcase my understanding of the topic. 

## Pledge

By submitting this work I hereby pledge that this is my own, personal work. I've acknowledged in the designated place at the top of this file all sources that I used to complete said work, including but not limited to: online resources, books, and electronic communications. I've noted all collaboration with fellow students and/or TA's. I did not copy or plagiarize another's work.

> As a Boilermaker pursuing academic excellence, I pledge to be honest and true in all that I do. Accountable together – We are Purdue.

https://www.purdue.edu/odos/osrr/honor-pledge/
